# PhiState-MoE: add code + general knowledge, then continue to 60K

This creates a **separate expanded corpus** from the original prepared corpus, keeps the existing 8,192-token BPE tokenizer and original validation/test bins, and continues from a verified 50K optimizer checkpoint. It adds up to 4M code tokens and 4M general-knowledge tokens, with a document-level 5% validation split for the new examples. The added-data source/license details are recorded in `provenance.json`.

The original corpus and checkpoint are never overwritten. The new run is saved in Google Drive and writes recovery checkpoints every 500 steps. If Colab disconnects, run this notebook again; it resumes from the last saved expanded-corpus checkpoint. Keep the runtime connected during a segment.

The code data is a small MIT-declared Python instruction/output dataset; general text is English Wikipedia, whose source material carries CC BY-SA/GFDL attribution terms. This is a modest extension, not a guarantee of better code correctness or general knowledge.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, importlib.util
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/phistate_l4')
REPO = ROOT / 'quantoniumoslinux'
LAB = REPO / 'runtime/experiments/phi_runtime_lab'
DRIVE = Path('/content/drive/MyDrive/Quantonium/PhiState-L4')
RUN_DRIVE = DRIVE / '50k'
BASE_DATA = DRIVE / 'data/hf_coding_knowledge_v2'
EXPANDED_DATA = DRIVE / 'data/hf_coding_knowledge_v2_expanded'

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Anharmoniclabs/quantoniumoslinux.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
if importlib.util.find_spec('datasets') is None or importlib.util.find_spec('sentencepiece') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'sentencepiece'], check=True)
assert (LAB / 'moe_data_extend.py').is_file() and (LAB / 'pretrain_continue_expanded.py').is_file(), 'Pull the updated main branch before continuing.'
assert BASE_DATA.joinpath('manifest.json').is_file(), f'Original corpus not found: {BASE_DATA}'
print('Repository:', REPO)
print('Original corpus:', BASE_DATA)
print('Expanded corpus will be:', EXPANDED_DATA)


In [ ]:
import torch

progress_path = RUN_DRIVE / 'progress.json'
assert progress_path.is_file(), 'Missing 50K progress.json in Google Drive.'
progress = json.loads(progress_path.read_text())
assert progress.get('step', 0) >= 50000, f'Requires the completed 50K checkpoint; current saved step is {progress.get("step")}'
slot = progress['active_slot']
CHECKPOINT = RUN_DRIVE / f'slot_{slot}.last.pt'
BEST = RUN_DRIVE / f'slot_{slot}.pt'
if not CHECKPOINT.is_file() or not BEST.is_file():
    CHECKPOINT = RUN_DRIVE / 'milestones/phi_50000.last.pt'
    BEST = RUN_DRIVE / 'milestones/phi_50000.pt'
assert CHECKPOINT.is_file() and BEST.is_file(), 'Verified 50K best/latest checkpoint pair not found.'
ck = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
assert ck['step'] >= 50000 and ck['data_fingerprint'] == progress['data_fingerprint']
assert 'optimizer_state' in ck, 'Optimizer state is required for exact continuation.'
print('Verified source step:', ck['step'])
print('Verified source fingerprint:', ck['data_fingerprint'])
print('Source checkpoint remains untouched:', CHECKPOINT)
del ck

# If an expanded run already exists, resume from its latest checkpoint.
OUT_BEST = RUN_DRIVE / 'phi_60000.pt'
OUT_LAST = RUN_DRIVE / 'phi_60000.last.pt'
if OUT_LAST.is_file():
    SOURCE_FOR_TRAINING = OUT_LAST
else:
    SOURCE_FOR_TRAINING = CHECKPOINT
print('Continuation starts from:', SOURCE_FOR_TRAINING)


In [ ]:
if not EXPANDED_DATA.joinpath('manifest.json').is_file():
    cmd = [sys.executable, '-u', str(LAB / 'moe_data_extend.py'),
           '--base', str(BASE_DATA), '--out', str(EXPANDED_DATA),
           '--max-code-tokens', '4000000', '--max-general-tokens', '4000000',
           '--validation-percent', '5']
    subprocess.run(cmd, check=True)
else:
    manifest = json.loads((EXPANDED_DATA / 'manifest.json').read_text())
    provenance = json.loads((EXPANDED_DATA / 'provenance.json').read_text())
    assert manifest['extension']['base_data_fingerprint'] == json.loads((BASE_DATA / 'manifest.json').read_text())['data_fingerprint']
    print('Reusing persisted expanded corpus:', EXPANDED_DATA)
    print('Added token counts:', {k: provenance['selection']['added'][k]['train'] + provenance['selection']['added'][k]['val'] for k in ('code', 'general')})


In [ ]:
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > L4 GPU and reconnect.'
cmd = [sys.executable, '-u', str(LAB / 'pretrain_continue_expanded.py'),
       '--checkpoint', str(SOURCE_FOR_TRAINING), '--data-dir', str(EXPANDED_DATA),
       '--out', str(OUT_BEST), '--target-steps', '60000', '--eval-every', '500',
       '--eval-batches', '16', '--threads', '2', '--device', 'cuda']
print('CONTINUING: existing trained weights -> 60,000 total steps on L4')
print('Best checkpoint:', OUT_BEST)
print('Resumable latest checkpoint:', OUT_LAST)
subprocess.run(cmd, check=True)


In [ ]:
if OUT_LAST.is_file():
    latest = torch.load(OUT_LAST, map_location='cpu', weights_only=True)
    best = torch.load(OUT_BEST, map_location='cpu', weights_only=True) if OUT_BEST.is_file() else {}
    print('SAVED STEP:', latest['step'], '/ 60000')
    print('NEW-VALIDATION BEST BPT:', best.get('best_validation_bpt'))
    print('EXPANDED FINGERPRINT:', latest['data_fingerprint'])
    print('Drive recovery checkpoint:', OUT_LAST)
    del latest, best
else:
    print('Training did not save a new checkpoint yet; inspect the error/output above.')
